# Multi-Omic Data Integration Workflow

### Author: Kaili Fan

## Introduction

This notebook provides a reproducible workflow for integrating post-QC RNA and ATAC multi-omic data, specifically using a multi-tissue 10X Multiome dataset. The pipeline guides you through removing remaining doublets, checking cell overlap, transferring cell annotations from RNA to ATAC, identifying potential cis-regulatory elements, and performing missing data imputation. Given tissue specificity, data should be processed on a per-tissue basis. 

To follow this guide, you will need post-QC, cell-annotated RNA Anndata and post-QC ATAC Anndata objects as input.

In [1]:
import sys
import os

repo_root = os.path.abspath(os.path.join("..")) 
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

It is recommended to start the workflow from the 'multi-Omic_workflow/multiome_integration' directory. This ensures that all relative paths and function imports work as intended. Alternatively, you may use absolute paths if preferred.

In [2]:
os.getcwd()

'/path/to/Multiomic-workflow/multiome_integration/notebooks'

In [3]:
os.chdir('../')
os.getcwd()

'/path/to/Multiomic-workflow/multiome_integration'

All log outputs are stored in log files within the logs folder.

In [4]:
os.makedirs("test_output/logs", exist_ok=True)

## Config overview
### YAML File for Storing Input Parameters
All parameters and file paths used in this workflow are stored in config/joint_multiomic_config.yaml

Please review this configuration file to ensure that all parameters are correctly specified and as expected before proceeding with the analysis.

In [5]:
from multiome_integration.utils import load_config

config_path = "config/joint_multiomic_config.yaml"
config = load_config(config_path)
import yaml
print(yaml.dump(config, default_flow_style=False))

color:
  donor_colors:
    DonorA: '#FBB30B'
    DonorB: '#8e02da'
  tissue_colors:
    Adipose - Subcutaneous: '#1d8c46'
    Adrenal Gland: '#33DD33'
    Kidney: '#a32015'
    Liver - Left Lobe: '#AABB66'
    Lung - Left Cranial Lobe: '#99FF00'
    Muscle - Gastrocnemius: '#9191ff'
    Muscle - Quadriceps: '#5e5eff'
    Thyroid: '#006600'
my_color_palette:
- '#c50909'
- '#ffa806'
- '#fbe95d'
- '#16b900'
- '#65c4ff'
- '#0e3aa9'
- '#8e02da'
- '#795548'
- '#9E9E9E'
- '#ff5b13'
- '#154f11'
- '#8bc34a'
- '#ff37a6'
- '#0062ff'
- '#00bfae'
- '#ea9c28'
- '#6e7f27'
- '#43e8e6'
- '#a4ff5a'
- '#e72c80'
params:
  celllineage_obs: cell_lineage
  celltype_obs: celltype_broad
  imputation: true
  n_features: 50000
  n_threads: 16
  peak_half_width: 150
  suffix: Muscle_Quadriceps
  tissue: Muscle - Quadriceps
paths:
  atac_doublet_dir: /path/to/Multiomic-workflow/ATAC_QC/test_output/doublet_filter_processing.demo_run
  atac_post_qc_h5ad: /path/to/Multiomic-workflow/ATAC_QC/test_output/atac_h5ad/Musc

Make sure your files are formatted similarly, and update the file paths in config/joint_multiomic_config.yaml to point to your data files.

You can also modify the reference files in the configuration to match your dataset if needed.

- If you want to process all samples in sample_metatable, set tissue: --- in the config file; otherwise, specify a tissue to work on.
- 'suffix' is used for file naming. Please make sure the strings do not contain spaces or special characters.
- You can specify your own tissue or donor colors in the config file if desired; otherwise, the script will automatically assign colors.

### Table for Sample Information
All sample information is stored in the metatable, which is used throughout all sections of the workflow. You may use a comprehensive metatable containing all of your data; however, this workflow will process samples per designated tissue.

Please ensure your metatable follows the required format and includes all necessary sample data. Below is an example of how this table should look:

In [6]:
import pandas as pd

sample_metadata = pd.read_csv('../ATAC_QC/test_data/test_metatable.txt', sep='\t')

sample_metadata.head()

,DonorA-GEX-01,DonorA-ATAC-01,Human,DonorA,Toddler,M,Muscle - Quadriceps
0,DonorB-GEX-02,DonorB-ATAC-02,Human,DonorB,Pre-adult,M,Muscle - Quadriceps


## Step1 - check overlapping cells
Here, we will first remove remaining doublets by checking doublet cell barcodes detected in other other modality. If you do not want this step for some reason, just not provide the doublet file.

In [7]:
!python scripts/check_overlapping_cells.py config/joint_multiomic_config.yaml

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/check_overlapping_cells.py config/joint_multiomic_config.yaml > logs/check_overlapping_cells.log 2>&1

2026-11-12 15:28:03 [INFO] Working tissue: Muscle - Quadriceps
2026-11-12 15:28:03 [INFO] RNA Samples: ['DonorA-GEX-01', 'DonorB-GEX-02']
2026-11-12 15:28:03 [INFO] ATAC Samples: ['DonorA-ATAC-01', 'DonorB-ATAC-02']


2026-11-12 15:28:09 [INFO] ATAC doublet directory found: /path/to/Multiomic-workflow/ATAC_QC/test_output/doublet_filter_processing.demo_run
2026-11-12 15:28:09 [INFO] checking sample DonorB-GEX-02: Muscle_Quadriceps


2026-11-12 15:28:09 [INFO] checking sample DonorA-GEX-01: Muscle_Quadriceps
2026-11-12 15:28:09 [INFO] Removing 67 remaining doublet cells for ATAC.


2026-11-12 15:28:09 [INFO] adata reduced from 4364 to 4339 cells.


2026-11-12 15:28:10 [INFO] RNA doublet directory found: /path/to/Multiomic-workflow/RNA_QC/test_output/filter_qc.demo_run
2026-11-12 15:28:10 [INFO] checking sample DonorA-ATAC-01: Muscle - Quadriceps
2026-11-12 15:28:10 [INFO] checking sample DonorB-ATAC-02: Muscle - Quadriceps


2026-11-12 15:28:10 [INFO] Removing 2083 remaining doublet cells for RNA.


2026-11-12 15:28:10 [INFO] adata reduced from 2047 to 1807 cells.


2026-11-12 15:28:15 [INFO] Calculating overlapping cells and generating Venn plots...
2026-11-12 15:28:15 [INFO] ========== Processing sample: DonorA-GEX-01 / DonorA-ATAC-01 (1/2) ==========


2026-11-12 15:28:16 [INFO] Venn diagram saved: /path/to/Multiomic-workflow/multiome_integration/test_output/overlap_figures/shared_cells_venn.DonorA.png
2026-11-12 15:28:16 [INFO] ========== Processing sample: DonorB-GEX-02 / DonorB-ATAC-02 (2/2) ==========


2026-11-12 15:28:17 [INFO] Venn diagram saved: /path/to/Multiomic-workflow/multiome_integration/test_output/overlap_figures/shared_cells_venn.DonorB.png


2026-11-12 15:28:18 [INFO] Overlap bar plot saved: /path/to/Multiomic-workflow/multiome_integration/test_output/overlap_figures/multiome_overlapPercent_vennBar.Muscle_Quadriceps.png
2026-11-12 15:28:18 [INFO] Overlap statistics complete. Figures saved in /path/to/Multiomic-workflow/multiome_integration/test_output/overlap_figures


## Step2 - ATAC cell annotation
In this step, we use GLUE (https://github.com/gao-lab/GLUE) to integrate post-QC RNA and ATAC data, thus transfer cell lables to all ATAC cells.

Note:

- This workflow uses some new functions in GLUE, which may not be available in the standard release. To ensure you have all the required features, please install GLUE using
      'pip install git+https://github.com/gao-lab/GLUE.git'

- If you have access to a GPU, this step can be significantly accelerated. Expected run time will be 15min-1hr, but could be longer if you have big dataset

In [8]:
!python scripts/annotate_ATAC_cells.py config/joint_multiomic_config.yaml

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/annotate_ATAC_cells.py config/joint_multiomic_config.yaml > logs/annotate_ATAC_cells.log 2>&1

2026-11-12 15:28:23 [INFO] torch.cuda.is_available(): True


2026-11-12 15:28:23 [INFO] torch.cuda.device_count(): 8
2026-11-12 15:28:23 [INFO] Using device: cuda
2026-11-12 15:28:23 [INFO] Output path set to: /path/to/Multiomic-workflow/multiome_integration/test_output/glue
2026-11-12 15:28:23 [INFO] ========== Processing tissue: Muscle - Quadriceps ==========


2026-11-12 15:28:31 [INFO] [ATAC] Standardized obs['tissue']: ['Muscle - Quadriceps']
2026-11-12 15:28:31 [INFO] RNA...
2026-11-12 15:28:31 [INFO] Preparing RNA AnnData for 'Muscle_Quadriceps' using cell type column 'celltype_broad' ...
2026-11-12 15:28:31 [INFO] Computing highly variable genes (HVGs) ...


2026-11-12 15:28:32 [INFO] Getting highly_variable
False    27918
True      4588
Name: count, dtype: int64 highly variable genes


2026-11-12 15:28:32 [INFO] Plotting RNA UMAP before GLUE integration ...


Figure(640x480)
2026-11-12 15:28:32 [INFO] Annotating genes using GTF: /path/to/Multiomic-workflow/ATAC_QC/ref/gencode.v47.primary_assembly.annotation.gtf ...


... (1,330 outputs left out)

2026-11-12 15:48:17 [INFO] Annotation and merging completed for 'Muscle_Quadriceps'.


2026-11-12 15:48:18 [INFO] maxp pruned
2026-11-12 15:48:18 [INFO] cmap pruned
2026-11-12 15:48:18 [INFO] kern dropped
2026-11-12 15:48:18 [INFO] post pruned
2026-11-12 15:48:18 [INFO] FFTM dropped
2026-11-12 15:48:18 [INFO] GPOS pruned
2026-11-12 15:48:18 [INFO] GSUB pruned
2026-11-12 15:48:18 [INFO] glyf pruned
2026-11-12 15:48:18 [INFO] Added gid0 to subset
2026-11-12 15:48:18 [INFO] Added first four glyphs to subset
2026-11-12 15:48:18 [INFO] Closing glyph list over 'MATH': 43 glyphs before
2026-11-12 15:48:18 [INFO] Glyph names: ['.notdef', '.null', 'A', 'C', 'E', 'F', 'I', 'J', 'M', 'N', 'P', 'R', 'S', 'T', 'V', 'a', 'b', 'c', 'd', 'e', 'f', 'five', 'g', 'h', 'hyphen', 'i', 'l', 'm', 'n', 'nonmarkingreturn', 'o', 'one', 'p', 'r', 's', 'seven', 'space', 't', 'two', 'u', 'w', 'y', 'zero']
2026-11-12 15:48:18 [INFO] Glyph IDs:   [0, 1, 2, 3, 16, 19, 20, 21, 24, 26, 36, 38, 40, 41, 44, 45, 48, 49, 51, 53, 54, 55, 57, 68, 69, 70, 71, 72, 73, 74, 75, 76, 79, 80, 81, 82, 83, 85, 86, 87, 

2026-11-12 15:48:18 [INFO] GPOS subsetted
2026-11-12 15:48:18 [INFO] GSUB subsetted
2026-11-12 15:48:18 [INFO] name subsetting not needed
2026-11-12 15:48:18 [INFO] glyf subsetted
2026-11-12 15:48:18 [INFO] head pruned
2026-11-12 15:48:18 [INFO] OS/2 Unicode ranges pruned: [0]
2026-11-12 15:48:18 [INFO] OS/2 CodePage ranges pruned: [0]
2026-11-12 15:48:18 [INFO] glyf pruned
2026-11-12 15:48:18 [INFO] GDEF pruned
2026-11-12 15:48:18 [INFO] GPOS pruned
2026-11-12 15:48:18 [INFO] GSUB pruned
2026-11-12 15:48:18 [INFO] name pruned


2026-11-12 15:48:18 [INFO] Finished GLUE for tissue 'Muscle_Quadriceps'.


## Step3 - Identify cis-Regulatory Elements

Here, we call ATAC peaks to represent putative cis-regulatory elements. Peaks are called separately for each cell cluster within each tissue, then merged so that overlapping peaks are consolidated into single representative regions. 

By default, each peak is resized to 300 bp, but you can specify a different peak width in the config/joint_multiomic_config.yaml

In [9]:
!python scripts/identify_regulatory_elements.py config/joint_multiomic_config.yaml

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/identify_regulatory_elements.py config/joint_multiomic_config.yaml > logs/identify_regulatory_elements.log 2>&1

2026-11-12 15:48:24 [INFO] Output path set to: /path/to/Multiomic-workflow/multiome_integration/test_output/atac_peak
2026-11-12 15:48:24 [INFO] references.chrom_size not set; using the built-in hg38 chromosome sizes
2026-11-12 15:48:24 [INFO] ========== Processing tissue: Muscle - Quadriceps ==========
2026-11-12 15:48:24 [INFO] Loading ATAC AnnData: /path/to/Multiomic-workflow/multiome_integration/test_output/h5ad/ATAC.GLUE.Muscle_Quadriceps.h5ad


2026-11-12 15:48:24 [INFO] Calling peaks by cluster for tissue: Muscle_Quadriceps (n_jobs=16)
2026-11-12 15:48:24 [INFO] Exporting fragments...


[4s] ████████████████████░░░░░░░░░░░░░░░░░░░░       1/2       (eta: 4s)         

[13s] ████████████████████████████████████████       2/2       (eta: 0s)        

2026-11-12 15:48:45 [INFO] Calling peaks...


  0%|                                                    | 0/11 [00:00<?, ?it/s]

... (34 outputs left out)

2026-11-12 15:52:28 [INFO] Selected 250000 features.


/path/to/conda/envs/workenv/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


2026-11-12 15:52:56 [INFO] Cell-by-peak matrix saved: /path/to/Multiomic-workflow/multiome_integration/test_output/h5ad/ATAC_cellByPeak.Muscle_Quadriceps.h5ad


Figure(640x480)
2026-11-12 15:52:56 [INFO] UMAP plot saved: ./figures/_ATAC_cellByPeak.celltype.Muscle_Quadriceps.png
2026-11-12 15:52:56 [INFO] Done with Muscle - Quadriceps



## Step4 - Data imputation (Optional)

This step is optional.

In this step, we use MultiVI from the scvi-tools package to impute all signals, including missing values, for all cells. 

Please note that the imputed signals are dense and may not fully represent true biological measurements. You should carefully consider whether to perform imputation and which set of signals is most appropriate for your analysis.

In [10]:
!python scripts/data_imputation.py config/joint_multiomic_config.yaml

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/data_imputation.py config/joint_multiomic_config.yaml > logs/data_imputation.log 2>&1

/path/to/conda/envs/workenv/lib/python3.10/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_categorical_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


INFO: Seed set to 0
2026-11-12 15:53:03 - INFO - Seed set to 0
2026-11-12 15:53:03 [INFO] scvi-tools version: 1.2.1


2026-11-12 15:53:03 [INFO] torch.cuda.is_available(): True
2026-11-12 15:53:03 [INFO] torch.cuda.device_count(): 8
2026-11-12 15:53:03 [INFO] Using device: cuda
2026-11-12 15:53:03 [INFO] Output path set to: /path/to/Multiomic-workflow/multiome_integration/test_output/multivi
2026-11-12 15:53:03 [INFO] ========== Processing tissue: Muscle - Quadriceps ==========
2026-11-12 15:53:03 [INFO] Loading RNA and ATAC AnnData for tissue: Muscle - Quadriceps


2026-11-12 15:53:04 [INFO] Integrating all data...
2026-11-12 15:53:04 [INFO] n(RNA): 4339, n(ATAC): 1807, n(shared): 1355, n(rna_only): 2984, n(atac_only): 452
2026-11-12 15:53:04 [INFO] Assembling MultiVI input AnnData ...
2026-11-12 15:53:04 [INFO] Extracting ATAC for 1355 cells


2026-11-12 15:53:04 [INFO] Extracting RNA for 1355 cells (with celltype_broad)


2026-11-12 15:53:04 [INFO] Extracting ATAC for 452 cells
2026-11-12 15:53:04 [INFO] Extracting RNA for 2984 cells (with celltype_broad)


... (331 outputs left out)

2026-11-12 16:05:22 [INFO] Imputing expression matrix ...


2026-11-12 16:05:34 [INFO] Wrote imputed RNA: MultiVI_impute_RNA.Muscle_Quadriceps.h5ad
2026-11-12 16:05:34 [INFO] Imputing accessibility matrix ...


2026-11-12 16:06:35 [INFO] Wrote imputed ATAC: MultiVI_impute_ATAC.Muscle_Quadriceps.h5ad


2026-11-12 16:06:35 [INFO] Done integration with Muscle - Quadriceps



## Step5 - Final integration

This step integrates the results from the previous steps and generates final h5ad files: one each for RNA-only, ATAC-only, and multiome cells. 

You can specify in the config/joint_multiomic_config.yaml whether you have imputed data. The imputed values will be stored in an additional layer alongside the raw counts.

In [11]:
!python scripts/data_integration.py config/joint_multiomic_config.yaml

# Here, we are running the script to display logs directly in the notebook.
# For best practice, you can use the command below to save all logs to a file for future reference:
#!python scripts/data_integration.py config/joint_multiomic_config.yaml > logs/data_integration.log 2>&1

2026-11-12 16:06:40 [INFO] ========== Processing tissue: Muscle - Quadriceps ==========
2026-11-12 16:06:40 [INFO] Loading AnnData for tissue: Muscle - Quadriceps


2026-11-12 16:06:42 [INFO] Propagating RNA/ATAC obs fields ...
2026-11-12 16:06:42 [INFO] Propagating sampleID
2026-11-12 16:06:42 [INFO] Propagating leiden
2026-11-12 16:06:42 [INFO] Propagating cellbarcode
2026-11-12 16:06:42 [INFO] Propagating total_counts
2026-11-12 16:06:42 [INFO] Propagating n_genes_by_counts
2026-11-12 16:06:42 [INFO] Propagating n_genes
2026-11-12 16:06:42 [INFO] Propagating pct_counts_mt
2026-11-12 16:06:42 [INFO] Propagating pct_counts_ribo
2026-11-12 16:06:42 [INFO] Propagating pct_counts_hb
2026-11-12 16:06:42 [INFO] Propagating pct_exon_reads
2026-11-12 16:06:42 [INFO] Propagating MALAT1_CPM
2026-11-12 16:06:42 [INFO] Propagating doublet_score
2026-11-12 16:06:42 [INFO] Propagating doublet_probabilities
2026-11-12 16:06:42 [INFO] Propagating donorID
2026-11-12 16:06:42 [INFO] Propagating sampleID
2026-11-12 16:06:42 [INFO] Propagating leiden
2026-11-12 16:06:42 [INFO] Propagating n_fragment
2026-11-12 16:06:42 [INFO] Propagating frac_dup
2026-11-12 16:06:4

2026-11-12 16:06:42 [INFO] Adding var column (rna): mean_counts
2026-11-12 16:06:42 [INFO] Adding var column (rna): pct_dropout_by_counts
2026-11-12 16:06:42 [INFO] Adding var column (rna): total_counts
2026-11-12 16:06:42 [INFO] Adding var column (rna): n_cells
2026-11-12 16:06:42 [INFO] Adding var column (rna): highly_variable
2026-11-12 16:06:42 [INFO] Adding var column (rna): means
2026-11-12 16:06:42 [INFO] Adding var column (rna): dispersions
2026-11-12 16:06:42 [INFO] Adding var column (rna): dispersions_norm
2026-11-12 16:06:42 [INFO] Adding uns key (rna): donorID_colors
2026-11-12 16:06:42 [INFO] Adding uns key (rna): hvg
2026-11-12 16:06:42 [INFO] Adding uns key (rna): leiden
2026-11-12 16:06:42 [INFO] Adding uns key (rna): leiden_colors
2026-11-12 16:06:42 [INFO] Adding uns key (rna): neighbors
2026-11-12 16:06:42 [INFO] Adding uns key (rna): pca
2026-11-12 16:06:42 [INFO] Adding uns key (rna): rank_genes_groups
2026-11-12 16:06:42 [INFO] Adding uns key (rna): umap
2026-11-1

2026-11-12 16:06:46 [INFO] Propagating layer: rawcounts


2026-11-12 16:06:58 [INFO] Integrating imputed expression and accessibility signal into combined AnnData ...


2026-11-12 16:06:58 [INFO] Imputed expression matched for 32497/32497 genes; imputed accessibility matched for 0/50000 peaks.
2026-11-12 16:06:58 [WARNING] Imputed accessibility is defined on a different peak set than the ATAC features in the combined object; unmatched peaks are left at 0 in 'imputation_signal'. The full imputed matrices are in MultiVI_impute_RNA/ATAC.<tissue>.h5ad.


... (10 outputs left out)

2026-11-12 16:08:34 [INFO] Wrote ATAC AnnData: /path/to/Multiomic-workflow/multiome_integration/test_output/h5ad/multiome_final.ATAC.Muscle_Quadriceps.h5ad


Figure(640x480)


Figure(640x480)


2026-11-12 16:08:34 [INFO] Finish the whole workflow! Hooray!!!


## Bash Script to Run the Entire Multiomic Data Integrtaion Workflow
Below is the bash script notebooks/run_multiome_integration_workflow.sh that you can run the entire workflow. Alternatively, you may modify the script to run only specific steps as needed.

In [12]:
# example bash command (run from the multiome_integration/ directory):
#nohup bash notebooks/run_multiome_integration_workflow.sh > run_multiome_integration_workflow.log 2>&1 &